<a href="https://colab.research.google.com/github/aysenurkck29/Web-Scraping-/blob/main/webscrapingKT%C3%9CN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================================================================
# KTÜN AKADEMİK PERSONEL SCRAPER (Google Colab için)
# TÜM BİRİMLER – MANUEL LİSTE İLE (ÇALIŞIR)
# ===================================================================

# ---------- HÜCRE 1: Kurulum ----------
!apt-get install -y tesseract-ocr
!pip install pytesseract pillow requests beautifulsoup4 lxml pandas openpyxl tqdm

# ---------- HÜCRE 2: Importlar ----------
import base64
import io
import re
import time
import random
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from PIL import Image
import pytesseract
import pandas as pd
from tqdm.auto import tqdm

BASE_URL = "https://www.ktun.edu.tr"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

session = requests.Session()
session.headers.update(HEADERS)


# ---------- HÜCRE 3: Yardımcı Fonksiyonlar ----------
def get_soup(url, retries=3, timeout=20):
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=timeout)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")
        except requests.RequestException as e:
            print(f"  [Uyarı] {url} için hata ({attempt+1}/{retries}): {e}")
            time.sleep(2)
    return None


def ocr_email_from_base64_img(img_tag):
    """
    <img src="data:image/bmp;base64,..."> etiketinden eposta adresini OCR ile okur.
    Görüntü boyutunu kontrol eder, çok büyükse ölçeklendirme oranını ayarlar.
    """
    if img_tag is None or not img_tag.get("src"):
        return ""

    src = img_tag["src"]
    if "base64," not in src:
        return ""

    try:
        b64_data = src.split("base64,")[1]
        img_bytes = base64.b64decode(b64_data)
        img = Image.open(io.BytesIO(img_bytes))
    except Exception as e:
        return f"DECODE_ERROR:{e}"

    # Gri tonlama
    img = img.convert("L")
    w, h = img.size

    # Eğer görüntü çok büyükse (örneğin 2000 pikselden geniş), ölçeklendirme oranını düşür
    max_dim = 2000
    if w > max_dim or h > max_dim:
        # Oranı koruyarak küçült
        ratio = min(max_dim / w, max_dim / h)
        new_w = int(w * ratio)
        new_h = int(h * ratio)
        img = img.resize((new_w, new_h), Image.LANCZOS)
        w, h = img.size
        # Ölçeklendirme faktörünü ayarla (küçük görüntüler için)
        scale = 2
    else:
        # Normal ölçeklendirme
        scale = 6

    img = img.resize((w * scale, h * scale), Image.LANCZOS)
    img = img.point(lambda p: 255 if p > 140 else 0)

    try:
        raw_text = pytesseract.image_to_string(
            img,
            config="--psm 7 -c tessedit_char_whitelist=abcdefghijklmnopqrstuvwxyz0123456789@._-"
        ).strip().lower()
    except pytesseract.TesseractError as e:
        # Tesseract hatası durumunda (örneğin görüntü çok büyük) atla
        return f"TESSERACT_ERROR:{e}"

    match = re.search(r"[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}", raw_text)
    if match:
        return match.group(0)
    return f"OCR_FAILED:{raw_text}"


def extract_phone(text_block):
    match = re.search(r"(0?\d{3}[\s.-]?\d{3}[\s.-]?\d{2}[\s.-]?\d{2}(?:\s*/\s*\d{2,5})?)", text_block)
    return match.group(0).strip() if match else ""


# ---------- HÜCRE 4: Personel Linklerini Çek ----------
def get_personnel_links(akademik_personel_url):
    soup = get_soup(akademik_personel_url)
    if soup is None:
        return []
    personnel = []
    for a in soup.select("a[href*='PersonelBilgi']"):
        href = a.get("href")
        name_title = a.get_text(strip=True)
        if href:
            personnel.append({
                "isim_unvan_liste": name_title,
                "detay_url": urljoin(BASE_URL, href)
            })
    return personnel


# ---------- HÜCRE 5: Personel Detay Parse ----------
def parse_personnel_detail(detail_url):
    soup = get_soup(detail_url)
    if soup is None:
        return None

    h6 = soup.find("h6")
    full_name_title = h6.get_text(strip=True) if h6 else ""

    title_prefixes = [
        "Prof. Dr.", "Doç. Dr.", "Dr. Öğr. Üyesi", "Öğr. Gör. Dr.",
        "Arş. Gör. Dr.", "Arş. Gör.", "Öğr. Gör.", "Okutman", "Uzman"
    ]
    unvan = ""
    isim = full_name_title
    for prefix in sorted(title_prefixes, key=len, reverse=True):
        if full_name_title.startswith(prefix):
            unvan = prefix
            isim = full_name_title[len(prefix):].strip()
            break

    h7_list = soup.find_all("h7")
    fakulte, bolum = "", ""
    if len(h7_list) >= 1:
        lines = [line.strip() for line in h7_list[0].get_text("\n").split("\n") if line.strip()]
        if len(lines) >= 1:
            fakulte = lines[0]
        if len(lines) >= 2:
            bolum = lines[1]

    full_page_text = soup.get_text(" ", strip=True)
    telefon = extract_phone(full_page_text)

    email = ""
    for h7 in h7_list:
        img_tag = h7.find("img")
        if img_tag is not None:
            email = ocr_email_from_base64_img(img_tag)
            if email and not email.startswith(("DECODE_ERROR", "OCR_FAILED")):
                break

    return {
        "isim": isim,
        "unvan": unvan,
        "fakulte": fakulte,
        "bolum": bolum,
        "telefon": telefon,
        "eposta": email,
        "detay_url": detail_url
    }


# ---------- HÜCRE 6: Ana Scraping ----------
def scrape_all(departments, delay_range=(0.5, 1.5)):
    all_results = []
    print(f"Toplam {len(departments)} bölüm/birim taranacak.\n")

    all_personnel_links = []
    for dept in tqdm(departments, desc="Bölümler taranıyor"):
        plinks = get_personnel_links(dept["url"])
        if not plinks:
            print(f"  [Uyarı] Personel bulunamadı: {dept['bolum_adi']} -> {dept['url']}")
        for p in plinks:
            p["bolum_kaynagi"] = dept["bolum_adi"]
        all_personnel_links.extend(plinks)
        time.sleep(random.uniform(*delay_range))

    seen = set()
    unique_personnel = []
    for p in all_personnel_links:
        if p["detay_url"] not in seen:
            seen.add(p["detay_url"])
            unique_personnel.append(p)

    print(f"\nToplam {len(unique_personnel)} benzersiz akademisyen bulundu.\n")

    for p in tqdm(unique_personnel, desc="Akademisyen detayları + OCR"):
        detail = parse_personnel_detail(p["detay_url"])
        if detail:
            detail["bolum_kaynagi"] = p.get("bolum_kaynagi", "")
            all_results.append(detail)
        time.sleep(random.uniform(*delay_range))

    df = pd.DataFrame(all_results)
    return df


# ---------- HÜCRE 7: Çalıştırma ----------
if __name__ == "__main__":
    # ===== MANUEL BİRİM LİSTESİ (SİZİN VERDİĞİNİZ + EKLENENLER) =====
    DEPARTMENTS = [
        # ---- Mühendislik ve Doğa Bilimleri Fakültesi ----
        {"bolum_adi": "Çevre Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=ntW3n9AwbbaSX0MwDUpFpQ=="},
        {"bolum_adi": "Elektrik-Elektronik Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=ZsobiwaZkLryYw8fWfQxOA=="},
        {"bolum_adi": "Endüstri Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=2H4mH3NLjgsVdUafOdEThQ=="},
        {"bolum_adi": "Harita Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=GtPC3+lY2w8m0+EOO75WAg=="},
        {"bolum_adi": "İnşaat Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=Ej3Dbzz3349oEgtqSjv5kA=="},
        {"bolum_adi": "Jeoloji Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=lmhBvI1WXmJSLDI9rbi+9w=="},
        {"bolum_adi": "Kimya Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=rgBRWcGggPYACfWedM49Qg=="},
        {"bolum_adi": "Maden Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=VfEFkIiPj06K5AegYdZxcg=="},
        {"bolum_adi": "Makine Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=G4/pl1sz9VHjeDFjkrMnwQ=="},
        {"bolum_adi": "Metalurji ve Malzeme Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=Tb5Pus4Dq9Rh5nCrqi4KlQ=="},
        {"bolum_adi": "Mühendislik Temel Bilimleri", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=Hzx/6wBtnxTwvyYqTFAiiQ=="},

        # ---- Mimarlık ve Tasarım Fakültesi ----
        {"bolum_adi": "Endüstri Ürünleri Tasarımı", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=AHYtTDjqYNURthutyZsoEw=="},
        {"bolum_adi": "İç Mimarlık", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=g7/720EPpazuh0oxDXh6GA=="},
        {"bolum_adi": "Mimarlık", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=G6659ogsx1G6K6szUz5SaA=="},
        {"bolum_adi": "Şehir ve Bölge Planlama", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=JaS8Z4oYnTqo6TGsH/zZbg=="},

        # ---- Bilgisayar ve Bilişim Bilimleri Fakültesi ----
        {"bolum_adi": "Bilgisayar Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=CjjgkdJ2kGZdNA6detUMmQ=="},
        {"bolum_adi": "Yapay Zeka ve Makine Öğrenmesi", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=14Nor138UPTTvDv6Apnukw=="},
        {"bolum_adi": "Yazılım Mühendisliği", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=SpbdtHOPmq3Vr7JBoHq7IA=="},

        # ---- Teknik Bilimler Meslek Yüksekokulu ----
        {"bolum_adi": "Bilgisayar Teknolojileri", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=rD1ANYCJhoZuYWDAndCQsw=="},
        {"bolum_adi": "El Sanatları", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=RoJ7TrIm+gQpnAu4YQ3aLA=="},
        {"bolum_adi": "Elektrik ve Enerji", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=SCuj/CqWl+7lloeAQEg/LQ=="},
        {"bolum_adi": "Elektronik ve Otomasyon", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=NjHpq7CBI+34Hn3/sWhJJQ=="},
        {"bolum_adi": "Gıda İşleme", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=jjzWfP/XQ2+/jpg+UjK4QQ=="},
        {"bolum_adi": "Görsel, İşitsel Teknikler ve Medya Yapımcılığı", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=4hbRl2X3Y0Ux3RU2z1tuOQ=="},
        {"bolum_adi": "İnşaat", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=CkW6BanFyGC9s11DBm29zQ=="},
        {"bolum_adi": "Kimya ve Kimyasal İşleme Teknolojileri", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=ToIQCBNXz/ssYEj4HPiOSg=="},
        {"bolum_adi": "Makine ve Metal Teknolojileri", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=1F64UYn5VMCcnj0XynkyDw=="},
        {"bolum_adi": "Malzeme ve Malzeme İşleme Teknolojileri", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=JTxbNitn/gvt/S3LKiyjRQ=="},
        {"bolum_adi": "Mimarlık ve Şehir Planlama", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=5MeK8SHrnv2FTvSjSm0RHg=="},
        {"bolum_adi": "Mülkiyet Koruma ve Güvenlik", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=97hw79tQCS1NDZpHVZLztw=="},
        {"bolum_adi": "Tekstil, Giyim, Ayakkabı ve Deri", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=bBrLpJmoS4BZn92VMtTryA=="},

        # ---- Yabancı Diller Yüksekokulu (SİZİN VERDİĞİNİZ DOĞRU LİNK) ----
        {"bolum_adi": "Yabancı Diller", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=czoj63FhUbkzPkwn7xw/2Q=="},

        # ---- Ortak Dersler Bölümü (DOĞRU LİNK) ----
        {"bolum_adi": "Ortak Dersler", "url": "https://www.ktun.edu.tr/tr/Birim/AkademikPersonel/?brm=L/NflMKAr69doecHl24/uA=="},
    ]

    # Scraping'i başlat
    df = scrape_all(DEPARTMENTS)

    # Sonuçları göster
    print("\nÖrnek veriler:")
    print(df.head(20))

    # Kaydet ve indir
    excel_path = "ktun_akademik_personel.xlsx"
    csv_path = "ktun_akademik_personel.csv"
    df.to_excel(excel_path, index=False)
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"\nKaydedildi: {excel_path} / {csv_path}")

    try:
        from google.colab import files
        files.download(excel_path)
    except ImportError:
        print("(Colab dışında çalıştırıldığı için otomatik indirme atlandı.)")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 56 not upgraded.
Toplam 33 bölüm/birim taranacak.



Bölümler taranıyor:   0%|          | 0/33 [00:00<?, ?it/s]


Toplam 496 benzersiz akademisyen bulundu.



Akademisyen detayları + OCR:   0%|          | 0/496 [00:00<?, ?it/s]


Örnek veriler:
                       isim           unvan  \
0              Bilgehan NAS       Prof. Dr.   
1       Dilek ERDİRENÇELEBİ       Prof. Dr.   
2                  Esra YEL       Prof. Dr.   
3         Mehmet Emin ARGUN       Prof. Dr.   
4              Şükrü DURSUN       Prof. Dr.   
5             Gülnihal KARA        Doç. Dr.   
6         Havva ATEŞ GÜLMEZ        Doç. Dr.   
7               Merve KALEM        Doç. Dr.   
8               Selim DOĞAN        Doç. Dr.   
9         Sezen KÜÇÜKÇONGAR        Doç. Dr.   
10           Süheyla TONGUR        Doç. Dr.   
11          Gamze GÖKTEPELİ  Dr. Öğr. Üyesi   
12        Mehmet TÜRKYILMAZ  Dr. Öğr. Üyesi   
13        Seçil TUTAR ÖKSÜZ  Dr. Öğr. Üyesi   
14              Taylan DOLU  Dr. Öğr. Üyesi   
15  Müberra Nur KILIÇARSLAN       Arş. Gör.   
16     Ahmet Afşin KULAKSIZ       Prof. Dr.   
17               Akif DURDU       Prof. Dr.   
18           Bayram AKDEMİR       Prof. Dr.   
19             Ercan YALDIZ       Prof. Dr. 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>